In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet34, ResNet34_Weights
from torchvision import transforms
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from skimage.util import random_noise
import glob
import random
from PIL import Image, ImageEnhance, ImageOps
import torchvision.transforms.functional as F
from tqdm.notebook import tqdm
import sys
import requests # Pentru descărcarea imaginii de test

# Setări specifice Kaggle
# OUTPUT_DIR trebuie să fie în /kaggle/working/ pentru a putea scrie fișiere
KAGGLE_WORKING_DIR = "/kaggle/working/"

In [ ]:
# --- FUNCȚII AJUTĂTOARE PENTRU ZGOMOT ---

def add_gaussian_noise(img_pil, var=0.01):
    img = np.array(img_pil) / 255.0
    noisy = random_noise(img, mode='gaussian', var=var)
    return Image.fromarray((noisy * 255).astype(np.uint8))

def add_salt_pepper_noise(img_pil, amount=0.05):
    img = np.array(img_pil) / 255.0
    noisy = random_noise(img, mode='s&p', amount=amount)
    return Image.fromarray((noisy * 255).astype(np.uint8))

def add_speckle_noise(img_pil, var=0.01):
    img = np.array(img_pil) / 255.0
    noisy = random_noise(img, mode='speckle', var=var)
    return Image.fromarray((noisy * 255).astype(np.uint8))

# --- DEFINIREA TRANSFORMĂRILOR DETERMINISTE ---

# 1. Geometrice
GEO_TRANSFORMS = {
    'geo_hflip': lambda img: F.hflip(img),
    'geo_vflip': lambda img: F.vflip(img),
    'geo_rot_90': lambda img: F.rotate(img, 90, fill=0),
    'geo_rot_180': lambda img: F.rotate(img, 180, fill=0),
    'geo_rot_270': lambda img: F.rotate(img, 270, fill=0),
    'geo_rot_15': lambda img: F.rotate(img, 15, fill=0),
    'geo_rot_neg_15': lambda img: F.rotate(img, -15, fill=0),
    'geo_shear_x_10': lambda img: img.transform(img.size, Image.AFFINE, (1, 0.1, 0, 0, 1, 0)),
    'geo_shear_y_10': lambda img: img.transform(img.size, Image.AFFINE, (1, 0, 0, 0.1, 1, 0)),
    'geo_shear_x_neg_10': lambda img: img.transform(img.size, Image.AFFINE, (1, -0.1, 0, 0, 1, 0)),
    'geo_shear_y_neg_10': lambda img: img.transform(img.size, Image.AFFINE, (1, 0, 0, -0.1, 1, 0)),
    'geo_translate_x_10': lambda img: img.transform(img.size, Image.AFFINE, (1, 0, 50, 0, 1, 0)),
    'geo_translate_y_10': lambda img: img.transform(img.size, Image.AFFINE, (1, 0, 0, 0, 1, 50)),
    'geo_crop_zoom_80': lambda img: F.center_crop(img, [int(s * 0.8) for s in img.size]),
    'geo_crop_zoom_90': lambda img: F.center_crop(img, [int(s * 0.9) for s in img.size]),
}

# 2. Culoare
COLOR_TRANSFORMS = {
    'color_brightness_high': lambda img: ImageEnhance.Brightness(img).enhance(1.5),
    'color_brightness_low': lambda img: ImageEnhance.Brightness(img).enhance(0.5),
    'color_contrast_high': lambda img: ImageEnhance.Contrast(img).enhance(1.5),
    'color_contrast_low': lambda img: ImageEnhance.Contrast(img).enhance(0.5),
    'color_saturation_high': lambda img: ImageEnhance.Color(img).enhance(1.5),
    'color_saturation_low': lambda img: ImageEnhance.Color(img).enhance(0.5),
    'color_sharpness_high': lambda img: ImageEnhance.Sharpness(img).enhance(2.0),
    'color_sharpness_low': lambda img: ImageEnhance.Sharpness(img).enhance(0.5),
    'color_hue_p0_1': lambda img: F.adjust_hue(img, 0.1),
    'color_hue_n0_1': lambda img: F.adjust_hue(img, -0.1),
    'color_grayscale': lambda img: ImageOps.grayscale(img).convert('RGB'),
    'color_solarize': lambda img: ImageOps.solarize(img, threshold=128),
    'color_posterize': lambda img: ImageOps.posterize(img, bits=4),
    'color_autocontrast': lambda img: ImageOps.autocontrast(img),
    'color_equalize': lambda img: ImageOps.equalize(img),
}

# 3. Zgomot
NOISE_TRANSFORMS = {
    'noise_gauss_0_01': lambda img: add_gaussian_noise(img, var=0.01),
    'noise_gauss_0_02': lambda img: add_gaussian_noise(img, var=0.02),
    'noise_gauss_0_05': lambda img: add_gaussian_noise(img, var=0.05),
    'noise_sp_0_01': lambda img: add_salt_pepper_noise(img, amount=0.01),
    'noise_sp_0_05': lambda img: add_salt_pepper_noise(img, amount=0.05),
    'noise_speckle_0_01': lambda img: add_speckle_noise(img, var=0.01),
}

In [ ]:
def run_augmentation_notebook(source_dir, output_dir):
    
    # Ajustează căile. Sursa poate fi în /kaggle/input/...
    img_dir = os.path.join(source_dir, 'images')
    gfdm_dir = os.path.join(source_dir, 'gazefixationsdensitymaps')
    
    out_img_dir = os.path.join(output_dir, 'images')
    out_gfdm_dir = os.path.join(output_dir, 'gazefixationsdensitymaps')

    # Creăm directoarele de output în /kaggle/working/
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_gfdm_dir, exist_ok=True)

    # Căutăm imaginile
    image_paths = sorted(glob.glob(os.path.join(img_dir, '*.png')))
    
    # Dacă nu găsim imagini, încercăm să căutăm recursiv (uneori structura e diferită pe Kaggle)
    if not image_paths:
        print(f"Căutare recursivă în {img_dir}...")
        image_paths = sorted(glob.glob(os.path.join(img_dir, '**', '*.png'), recursive=True))

    if not image_paths:
        print(f"EROARE CRITICĂ: Nu am găsit imagini .png în {img_dir}.")
        print(f"Verifică unde ai montat dataset-ul în Kaggle (de obicei /kaggle/input/NUME_DATASET).")
        return 0

    print(f"Am găsit {len(image_paths)} imagini originale. Începe augmentarea...")
    
    total_generated = 0
    
    for img_path in tqdm(image_paths, desc="Procesare imagini"):
        
        base_name = os.path.basename(img_path).replace('.png', '')
        
        # Încercăm să găsim harta de saliență corespunzătoare
        # Numele poate varia ușor, așa că construim calea
        gfdm_name = base_name.replace('_N_', '_GFDM_N_') + '.png'
        gfdm_path = os.path.join(gfdm_dir, gfdm_name)
        
        # Dacă nu există direct, o căutăm recursiv în folderul de hărți
        if not os.path.exists(gfdm_path):
             possible_maps = glob.glob(os.path.join(gfdm_dir, '**', gfdm_name), recursive=True)
             if possible_maps:
                 gfdm_path = possible_maps[0]
             else:
                 # tqdm.write(f"ATENȚIE: Harta {gfdm_name} lipsește. Se sare peste.")
                 continue

        try:
            img_x_orig = Image.open(img_path).convert('RGB')
            img_y_orig = Image.open(gfdm_path).convert('L')
        except Exception as e:
            tqdm.write(f"EROARE la încărcarea {base_name}: {e}")
            continue
            
        # Salvare Original
        img_x_orig.save(os.path.join(out_img_dir, f"{base_name}_aug_000_original.png"))
        img_y_orig.save(os.path.join(out_gfdm_dir, f"{gfdm_name.replace('.png', '')}_aug_000_original.png"))
        total_generated += 1
        
        # Aplicare Transformări
        all_transforms = [(GEO_TRANSFORMS, 'geo', True), 
                          (COLOR_TRANSFORMS, 'color', False), 
                          (NOISE_TRANSFORMS, 'noise', False)]

        for transform_set, prefix, is_geometric in all_transforms:
            for aug_name, transform_func in transform_set.items():
                try:
                    aug_x = transform_func(img_x_orig)

                    if is_geometric:
                        aug_y = transform_func(img_y_orig)
                        # Fix dimensiuni dacă e cazul (crop/shear)
                        if aug_x.size != img_x_orig.size:
                            aug_x = aug_x.resize(img_x_orig.size, Image.BILINEAR)
                            aug_y = aug_y.resize(img_y_orig.size, Image.NEAREST)
                    else:
                        aug_y = img_y_orig
                        
                    # Verificare existență pentru a nu suprascrie inutil
                    new_img_name = f"{base_name}_{aug_name}.png"
                    if not os.path.exists(os.path.join(out_img_dir, new_img_name)):
                        new_gfdm_name = f"{gfdm_name.replace('.png', '')}_{aug_name}.png"
                    
                        aug_x.save(os.path.join(out_img_dir, new_img_name))
                        aug_y.save(os.path.join(out_gfdm_dir, new_gfdm_name))
                        total_generated += 1
                    
                except Exception as e:
                    pass # Ignorăm erorile minore de augmentare

    return total_generated

In [ ]:
# --- CONFIGURARE DATASET ---
# Modifică linia de mai jos cu numele corect al datasetului tău din Kaggle
# Poți vedea calea exactă în panoul din dreapta "Data" -> copy file path
SOURCE_DIR_KAGGLE = "/kaggle/input/mexculture142-altered/MexCulture142_Altered" 

# Directorul de ieșire (trebuie să fie în /kaggle/working/)
OUTPUT_DIR_KAGGLE = "/kaggle/working/MexCulture142_Augmented"

# Executăm augmentarea
if os.path.exists(SOURCE_DIR_KAGGLE):
    print("Dataset sursă găsit. Începem augmentarea...")
    total_imgs = run_augmentation_notebook(SOURCE_DIR_KAGGLE, OUTPUT_DIR_KAGGLE)
    print(f"Augmentare completă. Total imagini: {total_imgs}")
else:
    print(f"NU s-a găsit datasetul la: {SOURCE_DIR_KAGGLE}")
    print("Te rog verifică calea în panoul 'Data' din dreapta.")
    # Creăm directoare goale pentru a nu crăpa codul următor
    os.makedirs(os.path.join(OUTPUT_DIR_KAGGLE, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR_KAGGLE, 'gazefixationsdensitymaps'), exist_ok=True)

In [ ]:
# --- Model & Dataset ---

class SaliencyModel(nn.Module):
    def __init__(self):
        super(SaliencyModel, self).__init__()
        base_model = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-2])
        self.saliency_head = nn.Sequential(
            nn.Conv2d(512, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(size=(32, 32), mode='bilinear', align_corners=False),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        features = self.feature_extractor(x)
        saliency_map = self.saliency_head(features)
        return saliency_map

class MexCultureDataset(Dataset):
    def __init__(self, root_dir, transform_input=None, target_size=(32, 32)):
        self.image_dir = os.path.join(root_dir, 'images')
        self.gfdm_dir = os.path.join(root_dir, 'gazefixationsdensitymaps')
        self.image_files = [f for f in os.listdir(self.image_dir) if f.endswith('.png')]
        self.transform_input = transform_input
        self.target_size = target_size
        self.transform_target = transforms.Compose([
            transforms.Resize(self.target_size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor() 
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        # Reconstruim numele hărții pe baza convenției de augmentare
        if "_aug_" in img_name:
            # ex: imagine_aug_geo.png -> imagine_GFDM_N_1_aug_geo.png
            # Trebuie să fim atenți la pattern-ul de înlocuire
            base_part = img_name.split("_aug_")[0] 
            aug_part = img_name.split("_aug_")[1]
            gfdm_base = base_part.replace('_N_', '_GFDM_N_')
            gfdm_name = f"{gfdm_base}_aug_{aug_part}"
        else:
            gfdm_name = img_name.replace('.png', '').replace('_N_', '_GFDM_N_') + '.png'
            
        gfdm_path = os.path.join(self.gfdm_dir, gfdm_name)
        
        input_image = Image.open(img_path).convert('RGB')
        target_gfdm_image = Image.open(gfdm_path).convert('L')
        
        if self.transform_input:
            input_tensor = self.transform_input(input_image)
        
        target_tensor = self.transform_target(target_gfdm_image)
        return input_tensor, target_tensor

# --- Procesul de Antrenare ---

# Parametrii
TRAIN_DATA_DIR = OUTPUT_DIR_KAGGLE # Folosim directorul cu datele augmentate
BATCH_SIZE = 16 # Pe Kaggle avem GPU bun, putem crește batch size
NUM_EPOCHS = 10
LEARNING_RATE = 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Dispozitiv antrenare: {DEVICE}")

input_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Verificăm dacă avem date generate
if len(os.listdir(os.path.join(TRAIN_DATA_DIR, 'images'))) > 0:
    train_dataset = MexCultureDataset(root_dir=TRAIN_DATA_DIR, transform_input=input_transform)
    # Pe Kaggle num_workers=2 sau 4 merge bine de obicei
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    
    print(f"Start Antrenare pe {len(train_dataset)} imagini...")
    
    model = SaliencyModel().to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        
        for inputs, targets in tqdm(train_loader, desc=f"Epoca {epoch+1}"):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        print(f"Epoca {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss / len(train_loader):.6f}")

    # Salvare model în /kaggle/working/
    torch.save(model.state_dict(), '/kaggle/working/saliency_resnet34.pth')
    print("Model salvat.")
else:
    print("Nu s-au găsit date pentru antrenare. Verifică pasul de augmentare.")

In [ ]:
# --- TESTARE PE IMAGINE CUSTOM (inferenta) ---

MODEL_PATH = '/kaggle/working/saliency_resnet34.pth'
OUTPUT_NAME = '/kaggle/working/custom_saliency_prediction.png'
UPSCALED_NAME = '/kaggle/working/predicted_saliency_map_upscaled.png'

# Funcție pentru a obține o imagine de test
def get_test_image():
    # Cale temporară pe Kaggle
    test_img_path = "/kaggle/working/test_image.jpg"
    
    # URL către o imagine publică (ex: o pictură sau o clădire)
    url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ec/Mona_Lisa%2C_by_Leonardo_da_Vinci%2C_from_C2RMF_retouched.jpg/402px-Mona_Lisa%2C_by_Leonardo_da_Vinci%2C_from_C2RMF_retouched.jpg"
    
    if not os.path.exists(test_img_path):
        print("Descărcare imagine de test...")
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()
            with open(test_img_path, 'wb') as out_file:
                out_file.write(response.content)
            print("Imagine descărcată.")
        except Exception as e:
            print(f"Eroare la descărcare: {e}")
            return None
    return test_img_path

# Setăm calea imaginii
CUSTOM_IMAGE_PATH = get_test_image()

if CUSTOM_IMAGE_PATH and os.path.exists(MODEL_PATH):
    print(f"Rulez predicția pe: {CUSTOM_IMAGE_PATH}")
    
    model = SaliencyModel().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    
    img_pil = Image.open(CUSTOM_IMAGE_PATH).convert('RGB')
    input_tensor = input_transform(img_pil).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        pred_tensor = model(input_tensor)
        
    pred_np = pred_tensor.squeeze().cpu().numpy()
    out_img = Image.fromarray((pred_np * 255).astype(np.uint8), 'L')
    out_img.save(OUTPUT_NAME)
    
    # Upscale și afișare
    out_upscaled = out_img.resize(img_pil.size, Image.Resampling.BILINEAR)
    out_upscaled.save(UPSCALED_NAME)
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.title("Original")
    plt.imshow(img_pil)
    
    plt.subplot(1, 3, 2)
    plt.title("Harta Saliență (Raw)")
    plt.imshow(out_img, cmap='gray')
    
    plt.subplot(1, 3, 3)
    plt.title("Saliență Suprapusă")
    plt.imshow(img_pil)
    plt.imshow(out_upscaled, cmap='jet', alpha=0.5) # Suprapunere cu heatmap
    plt.show()
    
else:
    print("Modelul sau imaginea de test lipsesc.")